In [ ]:
"""
SENTINEL TECH 10.0 — “QQQ-beater” growth engine (SIN IA, cash)

Objetivo explícito:
- Batir QQQ (benchmark de inversión) y usar el índice como termómetro de régimen.
- Evitar “operabilidad” en crisis estructurales (risk-off) sin matar el bull market.
- Adaptabilidad SIN overfitting: ensemble multi-horizonte + stops ATR + risk parity + régimen 3 estados.
- Validación fuerte: purged walk-forward (2 años test) + stress real + bootstrap por bloques + reportes y gráficas.

Benchmarks:
- QQQ (tradable, benchmark principal)
- ^NDX (conceptual, opcional; puede ser price index)
- SPY (mercado amplio)
- Buy&Hold del universo tech (equiponderado)

Requisitos:
pip install yfinance pandas numpy matplotlib

Uso:
- Corre tal cual. En la 1a ejecución descarga y cachea. En siguientes corre rápido.
- Revisa outputs/ para CSV.
"""

import warnings
warnings.filterwarnings("ignore")

import os
import json
import hashlib
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional


# =========================
# CONFIG
# =========================

@dataclass
class CostsConfig:
    commission: float = 0.001
    slippage: float = 0.0002
    apply_slippage: bool = True


@dataclass
class Tech10Config:
    capital_initial: float = 100000.0
    start: str = "2006-01-01"
    end: str = "2026-02-20"
    trading_days: int = 252
    rf_annual: float = 0.0

    # Universo Tech (tesis)
    universe: Tuple[str, ...] = ('AAPL','MSFT','NVDA','GOOGL','AMZN','META','AVGO','ASML','TSM','ADBE','NFLX','AMD')

    # Bench/regime tickers
    bench_qqq: str = "QQQ"     # benchmark principal
    bench_ndx: str = "^NDX"    # conceptual (opcional)
    bench_spy: str = "SPY"     # mercado

    # Cache
    cache_dir: str = "data_cache"

    # -------- SIGNALS (adaptabilidad sin sobreajuste) --------
    # Trend ensemble horizons (días)
    ema_spans: Tuple[int, ...] = (63, 126, 252)     # 3, 6, 12 meses
    # Momentum windows (días)
    mom_windows: Tuple[int, ...] = (63, 126, 252)   # multi-horizonte
    # Score weights (deben sumar 1 aprox)
    w_trend: float = 0.50
    w_mom: float = 0.30
    w_rel: float = 0.20  # fuerza relativa vs QQQ

    # Selección Top-K
    top_k: int = 5

    # Histeresis de selección (evita parpadeo excesivo)
    rebalance_freq: str = "W-FRI"  # rebalance semanal (estable y realista)

    # -------- POSITION SIZING --------
    # Risk parity / inverse vol por activo
    ivol_window: int = 63
    ivol_floor: float = 1e-6

    # Exposición global (vol targeting)
    vol_target_on: bool = True
    vol_target_ann: float = 0.20     # growth-grade (para batir QQQ)
    port_vol_window: int = 63
    max_exposure: float = 1.0        # sin leverage
    min_exposure: float = 0.20       # evita apagado total salvo CRISIS

    # -------- STOPS (dinámicos) --------
    atr_window: int = 14
    atr_mult: float = 3.0            # trailing = entry_max - atr_mult * ATR
    stop_on: bool = True
    allow_reentry: bool = True

    # -------- REGIME (3 estados, suave) --------
    # Regime driver: QQQ (más práctico para retornos totales)
    regime_use: bool = True
    regime_ma_fast: int = 50
    regime_ma_slow: int = 200
    # Crisis detection
    crisis_dd_threshold: float = 0.18    # DD del QQQ >= 18% => CRISIS
    crisis_vol_threshold: float = 0.30   # vol anual QQQ >= 30% => CRISIS (aprox)
    crisis_min_days: int = 5             # persistencia mínima
    # Escalas por régimen
    scale_risk_on: float = 1.00
    scale_caution: float = 0.55
    scale_crisis: float = 0.05           # “no operabilidad” casi total

    # -------- VALIDACIÓN --------
    cv_train_years: int = 8
    cv_test_years: int = 2
    purge_days: int = 10
    embargo_days: int = 5

    # score robusto (para evitar “pereza” extrema, usamos restricciones suaves)
    dd_cap_cv: float = -0.40
    lambda_dd: float = 1.5
    lambda_var: float = 0.6
    min_splits: int = 4


STRESS_EPISODES = {
    "GFC_2008": ("2008-01-01", "2009-06-30"),
    "EURO_2011": ("2011-07-01", "2011-12-31"),
    "Q4_2018": ("2018-10-01", "2018-12-31"),
    "COVID_2020": ("2020-02-15", "2020-06-30"),
    "RATES_2022": ("2022-01-01", "2022-12-31"),
}


# =========================
# HELPERS
# =========================

def to_1d_series(x, name="x") -> pd.Series:
    if x is None:
        return pd.Series(dtype=float, name=name)
    if isinstance(x, pd.Series):
        return x.rename(name)
    if isinstance(x, pd.DataFrame):
        if x.shape[1] == 0:
            return pd.Series(dtype=float, name=name)
        return x.iloc[:, 0].rename(name)
    arr = np.asarray(x)
    if arr.ndim == 2 and arr.shape[1] == 1:
        arr = arr.reshape(-1)
    return pd.Series(arr, name=name)

def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _hash_key(obj: dict) -> str:
    s = json.dumps(obj, sort_keys=True).encode("utf-8")
    return hashlib.md5(s).hexdigest()

def download_close_cached(tickers: List[str], start: str, end: str, cache_dir: str) -> pd.DataFrame:
    _ensure_dir(cache_dir)
    key = _hash_key({"tickers": tickers, "start": start, "end": end, "auto_adjust": True})
    path = os.path.join(cache_dir, f"prices_{key}.csv")

    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=True)

    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, group_by="ticker", progress=False)
    idx = pd.bdate_range(start=start, end=end)
    df = pd.DataFrame(index=idx)

    for t in tickers:
        try:
            if isinstance(raw, pd.DataFrame) and "Close" in raw.columns:
                s = raw["Close"]
                if isinstance(s, pd.DataFrame):
                    s = s[t] if t in s.columns else None
            else:
                s = raw[t]["Close"]
            if s is None:
                continue
            s = pd.Series(s).dropna()
            if s.empty:
                continue
            s = s.reindex(idx).ffill(limit=5)  # huecos cortos
            df[t] = s
        except Exception:
            continue

    df = df.dropna(how="all")
    df.to_csv(path)
    return df

def align_returns_no_fill(price_series: pd.Series, idx: pd.DatetimeIndex) -> pd.Series:
    p = to_1d_series(price_series, "p").reindex(idx)  # NO fill
    return p.pct_change()

def turnover(weights: pd.DataFrame) -> pd.Series:
    dw = weights.diff().abs().fillna(0.0)
    return 0.5 * dw.sum(axis=1)

def apply_costs_turnover(port_gross: pd.Series, weights: pd.DataFrame, costs: CostsConfig) -> pd.Series:
    to = turnover(weights)
    tc = to * (costs.commission + (costs.slippage if costs.apply_slippage else 0.0))
    return (port_gross - tc).replace([np.inf, -np.inf], 0.0).fillna(0.0)

# =========================
# METRICS
# =========================

def total_return(r: pd.Series) -> float:
    r = to_1d_series(r, "r").dropna()
    return float((1.0 + r).prod() - 1.0) if len(r) else 0.0

def cagr(r: pd.Series, td=252) -> float:
    r = to_1d_series(r, "r").dropna()
    if len(r) == 0:
        return 0.0
    tr = (1.0 + r).prod() - 1.0
    return float((1.0 + tr) ** (td / len(r)) - 1.0)

def ann_vol(r: pd.Series, td=252) -> float:
    r = to_1d_series(r, "r").dropna()
    sd = r.std(ddof=1)
    return float(sd * np.sqrt(td)) if sd == sd else np.nan

def sharpe(r: pd.Series, rf_ann=0.0, td=252) -> float:
    r = to_1d_series(r, "r").dropna()
    rf_d = (1.0 + rf_ann) ** (1.0 / td) - 1.0
    ex = r - rf_d
    sd = ex.std(ddof=1)
    return float(np.sqrt(td) * ex.mean() / sd) if sd and sd == sd else 0.0

def sortino(r: pd.Series, rf_ann=0.0, td=252) -> float:
    r = to_1d_series(r, "r").dropna()
    rf_d = (1.0 + rf_ann) ** (1.0 / td) - 1.0
    ex = r - rf_d
    dn = ex.copy()
    dn[dn > 0] = 0
    sd = dn.std(ddof=1)
    return float(np.sqrt(td) * ex.mean() / sd) if sd and sd == sd else 0.0

def max_dd(eq: pd.Series) -> float:
    eq = to_1d_series(eq, "eq").dropna()
    if len(eq) == 0:
        return 0.0
    dd = eq / eq.cummax() - 1.0
    return float(dd.min())

def calmar(r: pd.Series, eq: pd.Series, td=252) -> float:
    a = cagr(r, td)
    d = max_dd(eq)
    return float(a / abs(d)) if d != 0 else np.inf

def cvar(r: pd.Series, alpha=0.05) -> float:
    x = to_1d_series(r, "r").dropna().values
    if len(x) == 0:
        return np.nan
    q = np.quantile(x, alpha)
    tail = x[x <= q]
    return float(tail.mean()) if len(tail) else float(q)

def summarize(res: Dict, cfg: Tech10Config) -> Dict:
    r = to_1d_series(res["returns_net"], "r").replace([np.inf, -np.inf], np.nan).dropna()
    eq = to_1d_series(res["equity"], "eq").dropna()
    exp = to_1d_series(res.get("exposure", pd.Series(0, index=r.index)), "exp").reindex(r.index).fillna(0.0)
    to = to_1d_series(res.get("turnover", pd.Series(0, index=r.index)), "to").reindex(r.index).fillna(0.0)

    return {
        "FinalEquity": float(eq.iloc[-1]) if len(eq) else np.nan,
        "TotalReturn": total_return(r),
        "CAGR": cagr(r, cfg.trading_days),
        "AnnVol": ann_vol(r, cfg.trading_days),
        "Sharpe": sharpe(r, cfg.rf_annual, cfg.trading_days),
        "Sortino": sortino(r, cfg.rf_annual, cfg.trading_days),
        "MaxDD": max_dd(eq),
        "Calmar": calmar(r, eq, cfg.trading_days),
        "CVaR_5": cvar(r, 0.05),
        "CVaR_1": cvar(r, 0.01),
        "AvgExposure": float(exp.mean()),
        "TimeInMkt": float((exp > 0).mean()),
        "TurnoverAnn": float(to.sum() * (cfg.trading_days / len(r))) if len(r) else 0.0,
        "Days": int(len(r)),
    }

# =========================
# REGIME (3-state)
# =========================

def compute_regime_scale(qqq_price: pd.Series, cfg: Tech10Config) -> pd.Series:
    """
    3 estados:
    - RISK_ON:  qqq > ma_slow (y/o ma_fast>ma_slow)
    - CAUTION:  condición intermedia
    - CRISIS:   DD grande o vol alta persistente
    Retorna escala en {1.0, 0.55, 0.05}.
    """
    p = to_1d_series(qqq_price, "QQQ").copy()
    idx = p.index

    ma_f = p.ewm(span=cfg.regime_ma_fast, adjust=False).mean()
    ma_s = p.ewm(span=cfg.regime_ma_slow, adjust=False).mean()

    # DD del índice
    dd = p / p.cummax() - 1.0

    # vol anual aproximada del índice
    r = p.pct_change().fillna(0.0)
    vol = r.rolling(cfg.port_vol_window).std() * np.sqrt(cfg.trading_days)
    vol = vol.fillna(method="bfill").fillna(method="ffill").fillna(0.0)

    crisis_raw = ((dd <= -cfg.crisis_dd_threshold) | (vol >= cfg.crisis_vol_threshold)).astype(int)
    crisis = crisis_raw.rolling(cfg.crisis_min_days).mean().fillna(0.0) >= 0.8

    scale = pd.Series(cfg.scale_risk_on, index=idx, dtype=float)

    risk_on = (p > ma_s) & (ma_f >= ma_s)
    caution = ~risk_on

    scale[caution] = cfg.scale_caution
    scale[crisis] = cfg.scale_crisis

    # burn-in conservador
    scale.iloc[:cfg.regime_ma_slow] = cfg.scale_crisis
    return scale

# =========================
# SIGNALS (trend + momentum + relative strength)
# =========================

def compute_scores(prices: pd.DataFrame, qqq_price: pd.Series, cfg: Tech10Config) -> pd.DataFrame:
    """
    score = w_trend*trend + w_mom*mom + w_rel*rel_strength
    Todas 0..1 por activo.
    """
    idx = prices.index
    scores = pd.DataFrame(index=idx, columns=prices.columns, dtype=float)

    # qqq returns for relative strength
    qqq_r = align_returns_no_fill(qqq_price, idx).fillna(0.0)

    for t in prices.columns:
        p = prices[t]

        # Trend votes: price > EMA(span) shifted 1
        votes = []
        for span in cfg.ema_spans:
            ema = p.ewm(span=span, adjust=False).mean().shift(1)
            v = ((p > ema) & p.notna() & ema.notna()).astype(float)
            votes.append(v)
        trend = sum(votes) / len(votes)
        trend.iloc[:cfg.burn_in] = 0.0

        # Momentum: multi-window returns (shifted 1 to avoid same-day)
        moms = []
        for w in cfg.mom_windows:
            mom = (p / p.shift(w) - 1.0).shift(1)
            moms.append(mom)
        mom_raw = sum(moms) / len(moms)
        # Map to 0..1 via sigmoid-ish normalization (robust)
        mom = (mom_raw.clip(-1, 1) + 1.0) / 2.0  # -1..1 -> 0..1
        mom.iloc[:max(cfg.mom_windows)] = 0.0

        # Relative strength: (asset rolling return - QQQ rolling return)
        rels = []
        for w in (63, 126):
            a = (p / p.shift(w) - 1.0).shift(1)
            b = ((1.0 + qqq_r).rolling(w).apply(np.prod, raw=True) - 1.0).shift(1)
            rels.append(a - b)
        rel_raw = sum(rels) / len(rels)
        rel = (rel_raw.clip(-1, 1) + 1.0) / 2.0
        rel.iloc[:200] = 0.0

        s = cfg.w_trend * trend + cfg.w_mom * mom + cfg.w_rel * rel
        scores[t] = s.fillna(0.0)

    return scores.fillna(0.0)

# =========================
# SELECTION + WEIGHTS (Top-K + inverse-vol)
# =========================

def select_topk(scores: pd.DataFrame, k: int, freq: str) -> pd.DataFrame:
    """
    Selección discreta Top-K (se actualiza en fechas de rebalance).
    Retorna mask 0/1 por activo.
    """
    idx = scores.index
    reb_dates = scores.resample(freq).last().index
    mask = pd.DataFrame(0.0, index=idx, columns=scores.columns)

    last = np.zeros(scores.shape[1])
    cols = list(scores.columns)

    for dt in idx:
        if dt in reb_dates:
            row = scores.loc[dt].values
            # top-k por score
            order = np.argsort(-row)
            top = set(order[:k])
            last = np.array([1.0 if i in top else 0.0 for i in range(len(cols))], dtype=float)
        mask.loc[dt] = last

    return mask.fillna(0.0)

def inverse_vol_weights(prices: pd.DataFrame, active_mask: pd.DataFrame, window: int, floor: float) -> pd.DataFrame:
    """
    Pesos ∝ 1/vol_i dentro del set activo.
    """
    rets = prices.pct_change().fillna(0.0)
    vol = rets.rolling(window).std().replace(0, np.nan).fillna(method="bfill").fillna(method="ffill")
    iv = 1.0 / (vol + floor)

    w_raw = iv * active_mask
    denom = w_raw.sum(axis=1).replace(0, np.nan)
    w = w_raw.div(denom, axis=0).fillna(0.0)
    return w

# =========================
# ATR Trailing Stop (dynamic)
# =========================

def atr(prices: pd.Series, window: int) -> pd.Series:
    """
    Aproximación ATR con closes: ATR ~ rolling mean of abs returns * price
    (Sin OHLC; suficiente como stop dinámico para este prototipo)
    """
    p = prices
    tr = p.pct_change().abs() * p
    return tr.rolling(window).mean()

def apply_atr_trailing_stop(prices: pd.DataFrame, w: pd.DataFrame, cfg: Tech10Config) -> pd.DataFrame:
    """
    Aplica stop por activo:
    - si activo está en cartera (peso >0), trackea máximo desde entrada
    - stop level = max_price - atr_mult * ATR
    - si price < stop -> fuerza peso 0 hasta próximo rebalance (o reentrada si allow_reentry)
    Implementación: stop actúa sobre el mask/weights en el día (no look-ahead: se aplica para el día siguiente vía shift).
    """
    if not cfg.stop_on:
        return w

    out = w.copy()
    idx = out.index
    cols = out.columns
    reb_dates = out.resample(cfg.rebalance_freq).last().index

    for t in cols:
        p = prices[t].reindex(idx).ffill()
        a = atr(p, cfg.atr_window).reindex(idx).fillna(method="bfill").fillna(0.0)
        wt = out[t].values

        in_pos = False
        maxp = np.nan
        stopped = False
        for i, dt in enumerate(idx):
            # rebalance date resets ability to enter (if allow_reentry)
            if dt in reb_dates and cfg.allow_reentry:
                stopped = False

            if wt[i] <= 0:
                in_pos = False
                maxp = np.nan
                continue

            # we have weight > 0
            if stopped:
                wt[i] = 0.0
                in_pos = False
                maxp = np.nan
                continue

            if not in_pos:
                in_pos = True
                maxp = float(p.iloc[i])
            else:
                maxp = max(maxp, float(p.iloc[i]))

            stop_level = maxp - cfg.atr_mult * float(a.iloc[i])
            if float(p.iloc[i]) < stop_level:
                wt[i] = 0.0
                stopped = True
                in_pos = False
                maxp = np.nan

        out[t] = wt

    # Renormaliza pesos para que sumen 1 cuando haya activos
    denom = out.sum(axis=1).replace(0, np.nan)
    out = out.div(denom, axis=0).fillna(0.0)
    return out

# =========================
# VOL TARGET (portfolio)
# =========================

def vol_target_scale(port_gross: pd.Series, cfg: Tech10Config) -> pd.Series:
    if not cfg.vol_target_on:
        return pd.Series(1.0, index=port_gross.index)
    r = to_1d_series(port_gross, "r").fillna(0.0)
    vol = r.rolling(cfg.port_vol_window).std() * np.sqrt(cfg.trading_days)
    scale = (cfg.vol_target_ann / vol).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    scale = scale.clip(lower=cfg.min_exposure, upper=cfg.max_exposure)
    return scale

# =========================
# BACKTEST TECH10
# =========================

def backtest_tech10(prices: pd.DataFrame, qqq_price: pd.Series, cfg: Tech10Config, costs: CostsConfig) -> Dict:
    idx = prices.index

    # 1) Regime scale (3-state)
    if cfg.regime_use:
        regime_scale = compute_regime_scale(qqq_price.reindex(idx), cfg)
    else:
        regime_scale = pd.Series(1.0, index=idx)

    # 2) Scores
    scores = compute_scores(prices, qqq_price.reindex(idx), cfg)

    # 3) Top-K selection on rebalance dates
    active_mask = select_topk(scores, cfg.top_k, cfg.rebalance_freq)

    # 4) Inverse-vol weights within active set
    w = inverse_vol_weights(prices, active_mask, cfg.ivol_window, cfg.ivol_floor)

    # 5) ATR trailing stop (dynamic)
    w = apply_atr_trailing_stop(prices, w, cfg)

    # 6) Gross portfolio returns (weights shifted 1 to avoid look-ahead)
    rets = prices.pct_change().fillna(0.0)
    port_gross = (w.shift(1).fillna(0.0) * rets).sum(axis=1)

    # 7) Vol target scale (cash)
    vol_scale = vol_target_scale(port_gross, cfg)

    # 8) Combine scales: regime * vol_target
    total_scale = (regime_scale * vol_scale).clip(lower=0.0, upper=cfg.max_exposure)

    w_scaled = w.mul(total_scale, axis=0)

    # 9) Costs via turnover on scaled weights
    port_net = apply_costs_turnover(port_gross * total_scale, w_scaled, costs)

    equity = cfg.capital_initial * (1.0 + port_net).cumprod()
    exposure = w_scaled.abs().sum(axis=1).clip(0, cfg.max_exposure)

    return {
        "returns_net": port_net,
        "equity": equity,
        "scores": scores,
        "active_mask": active_mask,
        "weights": w,
        "weights_scaled": w_scaled,
        "regime_scale": regime_scale,
        "vol_scale": vol_scale,
        "total_scale": total_scale,
        "exposure": exposure,
        "turnover": turnover(w_scaled),
    }

def buy_hold_equal(prices: pd.DataFrame, cfg: Tech10Config) -> Dict:
    first = prices.iloc[0]
    cols = [c for c in prices.columns if not pd.isna(first[c])]
    if len(cols) == 0:
        cols = [c for c in prices.columns if prices[c].notna().any()]
    sub = prices[cols]
    w0 = pd.Series(1.0 / len(cols), index=cols)
    r = sub.pct_change().fillna(0.0)
    port = (r * w0).sum(axis=1)
    eq = cfg.capital_initial * (1.0 + port).cumprod()
    return {"returns_net": port, "equity": eq, "exposure": pd.Series(1.0, index=port.index), "turnover": pd.Series(0.0, index=port.index)}

# =========================
# VALIDATION: Purged WF CV
# =========================

def purged_wf_splits(index: pd.DatetimeIndex, cfg: Tech10Config) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    years = sorted(index.year.unique())
    out = []
    for i in range(cfg.cv_train_years, len(years) - cfg.cv_test_years + 1):
        train_years = years[i - cfg.cv_train_years:i]
        test_years = years[i:i + cfg.cv_test_years]
        tr = index[index.year.isin(train_years)]
        te = index[index.year.isin(test_years)]
        if len(tr) == 0 or len(te) == 0:
            continue

        test_start = te.min()
        if len(tr) > cfg.purge_days:
            tr = tr[:-cfg.purge_days]
        if len(te) > cfg.purge_days:
            te = te[cfg.purge_days:]

        embargo_cut = test_start - pd.tseries.offsets.BDay(cfg.embargo_days)
        tr = tr[tr <= embargo_cut]

        if len(tr) < 300 or len(te) < 300:
            continue
        out.append((tr, te))
    return out

def score_cv(prices: pd.DataFrame, qqq_price: pd.Series, cfg: Tech10Config, costs: CostsConfig) -> Dict:
    splits = purged_wf_splits(prices.index, cfg)
    if len(splits) < cfg.min_splits:
        return {"score": -np.inf, "avg_sharpe": np.nan, "std_sharpe": np.nan, "worst_dd": np.nan, "splits": len(splits)}

    sharpes_list, dd_list = [], []
    # folds con burn-in reducido para no matar ventanas
    fold_cfg = Tech10Config(**{**cfg.__dict__, "burn_in": min(cfg.burn_in, 80)})

    for _, te in splits:
        res = backtest_tech10(prices.loc[te], qqq_price.loc[te], fold_cfg, costs)
        s = summarize(res, fold_cfg)
        sharpes_list.append(s["Sharpe"])
        dd_list.append(s["MaxDD"])

    avg_sh = float(np.nanmean(sharpes_list))
    std_sh = float(np.nanstd(sharpes_list))
    worst_dd = float(np.nanmin(dd_list))

    dd_excess = max(0.0, abs(worst_dd) - abs(cfg.dd_cap_cv))
    score = avg_sh - cfg.lambda_dd * dd_excess - cfg.lambda_var * std_sh

    return {"score": score, "avg_sharpe": avg_sh, "std_sharpe": std_sh, "worst_dd": worst_dd, "splits": len(splits)}

# =========================
# STRESS + BOOTSTRAP
# =========================

def stress_report(prices: pd.DataFrame, qqq_price: pd.Series, cfg: Tech10Config, costs: CostsConfig,
                  episodes: Dict[str, Tuple[str, str]]) -> pd.DataFrame:
    rows = []
    for name, (a, b) in episodes.items():
        sub = prices.loc[a:b]
        if len(sub) < 40:
            continue
        ep_cfg = Tech10Config(**{**cfg.__dict__, "burn_in": 0})  # stress: no burn-in
        res = backtest_tech10(sub, qqq_price.loc[sub.index], ep_cfg, costs)
        s = summarize(res, ep_cfg)
        rows.append({
            "Episode": name,
            "CAGR%": round(s["CAGR"] * 100, 2),
            "Sharpe": round(s["Sharpe"], 3),
            "MaxDD%": round(s["MaxDD"] * 100, 2),
            "Calmar": round(s["Calmar"], 3),
            "AvgExposure%": round(s["AvgExposure"] * 100, 1),
            "TimeInMkt%": round(s["TimeInMkt"] * 100, 1),
        })
    return pd.DataFrame(rows)

def moving_block_bootstrap(r: pd.Series, block=20, n_samples=500, seed=42) -> Dict:
    rng = np.random.default_rng(seed)
    x = to_1d_series(r, "r").replace([np.inf, -np.inf], np.nan).dropna().values
    T = len(x)
    if T < block * 5:
        return {"dd_p50": np.nan, "dd_p95": np.nan, "ruin_prob_50dd": np.nan}
    dds = []
    for _ in range(n_samples):
        starts = rng.integers(0, T - block, size=int(np.ceil(T / block)))
        sample = np.concatenate([x[s:s + block] for s in starts])[:T]
        eq = (1.0 + sample).cumprod()
        peak = np.maximum.accumulate(eq)
        dds.append(np.min(eq / peak - 1.0))
    dds = np.array(dds)
    return {
        "dd_p50": float(np.quantile(dds, 0.50)),
        "dd_p95": float(np.quantile(dds, 0.05)),
        "ruin_prob_50dd": float(np.mean(dds < -0.5)) * 100.0
    }

# =========================
# PLOTS (explicativas)
# =========================

def plot_equity(curves: Dict[str, pd.Series], title: str):
    plt.figure()
    for k, s in curves.items():
        s = to_1d_series(s, k).dropna()
        if len(s):
            plt.plot(s.index, s.values, label=k)
    plt.title(title); plt.xlabel("Date"); plt.ylabel("Equity"); plt.legend(); plt.show()

def plot_drawdown(eq: pd.Series, title: str):
    eq = to_1d_series(eq, "eq").dropna()
    dd = eq / eq.cummax() - 1.0
    plt.figure()
    plt.plot(dd.index, dd.values)
    plt.title(title); plt.xlabel("Date"); plt.ylabel("Drawdown"); plt.show()

def plot_scales(res: Dict, title: str):
    exp = to_1d_series(res["exposure"], "exp").fillna(0)
    reg = to_1d_series(res["regime_scale"], "reg").fillna(0)
    vol = to_1d_series(res["vol_scale"], "vol").fillna(0)
    tot = to_1d_series(res["total_scale"], "tot").fillna(0)

    plt.figure()
    plt.plot(exp.index, exp.values, label="Exposure")
    plt.plot(reg.index, reg.values, label="RegimeScale")
    plt.plot(vol.index, vol.values, label="VolScale")
    plt.plot(tot.index, tot.values, label="TotalScale")
    plt.title(title); plt.xlabel("Date"); plt.legend(); plt.show()

def plot_turnover(to: pd.Series, title: str):
    to = to_1d_series(to, "to").fillna(0)
    plt.figure()
    plt.plot(to.index, to.values)
    plt.title(title); plt.xlabel("Date"); plt.ylabel("Turnover"); plt.show()

def plot_rolling_metrics(r: pd.Series, title: str, window=126):
    r = to_1d_series(r, "r").replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < window * 2:
        return
    roll_sh = (r.rolling(window).mean() / r.rolling(window).std()).replace([np.inf, -np.inf], np.nan) * np.sqrt(252)
    roll_vol = r.rolling(window).std() * np.sqrt(252)
    plt.figure(); plt.plot(roll_sh.index, roll_sh.values); plt.title(f"Rolling Sharpe ({window}d) — {title}")
    plt.xlabel("Date"); plt.ylabel("Sharpe"); plt.show()
    plt.figure(); plt.plot(roll_vol.index, roll_vol.values); plt.title(f"Rolling Vol ({window}d) — {title}")
    plt.xlabel("Date"); plt.ylabel("AnnVol"); plt.show()

def plot_year_bars(r: pd.Series, title: str):
    yr = (1.0 + r.fillna(0.0)).resample("Y").prod() - 1.0
    if len(yr) == 0:
        return
    df = pd.DataFrame({"Year": yr.index.year, "Ret%": (yr.values * 100)})
    colors = ["green" if x >= 0 else "red" for x in df["Ret%"]]
    plt.figure()
    plt.bar(df["Year"], df["Ret%"], color=colors)
    plt.axhline(0, linewidth=0.8)
    plt.title(f"Yearly Returns (%) — {title}")
    plt.xlabel("Year"); plt.ylabel("Return %")
    plt.show()

def print_monthly_table(r: pd.Series, title: str):
    mr = (1.0 + r.fillna(0.0)).resample("M").prod() - 1.0
    if len(mr) == 0:
        return
    df = pd.DataFrame({"date": mr.index, "ret": mr.values})
    df["Year"] = df["date"].dt.year
    df["Month"] = df["date"].dt.month
    piv = df.pivot_table(index="Year", columns="Month", values="ret", aggfunc="mean") * 100
    print("\nMonthly return table (%):", title)
    print(piv.round(2).to_string())

def plot_participation(res: Dict, qqq_r: pd.Series, title: str, window=63):
    """
    Participación en rebotes: correlación con QQQ y ratio de capturas (up/down capture).
    """
    strat = to_1d_series(res["returns_net"], "s").fillna(0.0)
    mkt = to_1d_series(qqq_r, "m").reindex(strat.index).fillna(0.0)

    up = mkt[mkt > 0]
    dn = mkt[mkt < 0]
    up_capture = (strat.loc[up.index].mean() / up.mean()) if up.mean() != 0 else np.nan
    dn_capture = (strat.loc[dn.index].mean() / dn.mean()) if dn.mean() != 0 else np.nan

    roll_corr = strat.rolling(window).corr(mkt)

    plt.figure()
    plt.plot(roll_corr.index, roll_corr.values)
    plt.title(f"Rolling Corr vs QQQ ({window}d) — {title}")
    plt.xlabel("Date"); plt.ylabel("Corr")
    plt.show()

    print(f"\nParticipation stats — {title}")
    print(f"  Up-capture (mean strat / mean QQQ on up days):  {up_capture:.3f}")
    print(f"  Down-capture (mean strat / mean QQQ on down days): {dn_capture:.3f}")

# =========================
# RUNNER
# =========================

def run_tech10(make_plots: bool = True):
    cfg = Tech10Config()
    costs = CostsConfig()

    # Download once (universe + benchmarks)
    tickers = list(cfg.universe) + [cfg.bench_qqq, cfg.bench_ndx, cfg.bench_spy]
    data = download_close_cached(tickers, cfg.start, cfg.end, cfg.cache_dir)

    prices = data[list(cfg.universe)].copy()
    qqq_price = data[cfg.bench_qqq].copy()
    ndx_price = data[cfg.bench_ndx].copy()
    spy_price = data[cfg.bench_spy].copy()

    # Bench returns/equity
    qqq_r = align_returns_no_fill(qqq_price, prices.index).fillna(0.0)
    ndx_r = align_returns_no_fill(ndx_price, prices.index).fillna(0.0)
    spy_r = align_returns_no_fill(spy_price, prices.index).fillna(0.0)

    qqq_eq = cfg.capital_initial * (1.0 + qqq_r).cumprod()
    ndx_eq = cfg.capital_initial * (1.0 + ndx_r).cumprod()
    spy_eq = cfg.capital_initial * (1.0 + spy_r).cumprod()

    # Sentinel Tech10
    res = backtest_tech10(prices, qqq_price, cfg, costs)

    # B&H Tech
    bh = buy_hold_equal(prices, cfg)

    # Summaries
    s_sent = summarize(res, cfg)
    s_bh = summarize(bh, cfg)
    s_qqq = summarize({"returns_net": qqq_r, "equity": qqq_eq, "exposure": pd.Series(1.0, index=qqq_r.index), "turnover": pd.Series(0.0, index=qqq_r.index)}, cfg)
    s_ndx = summarize({"returns_net": ndx_r, "equity": ndx_eq, "exposure": pd.Series(1.0, index=ndx_r.index), "turnover": pd.Series(0.0, index=ndx_r.index)}, cfg)
    s_spy = summarize({"returns_net": spy_r, "equity": spy_eq, "exposure": pd.Series(1.0, index=spy_r.index), "turnover": pd.Series(0.0, index=spy_r.index)}, cfg)

    # Comparison table
    def row(name, s):
        return {
            "Model": name,
            "FinalEq": round(s["FinalEquity"], 2),
            "Total%": round(s["TotalReturn"]*100, 2),
            "CAGR%": round(s["CAGR"]*100, 2),
            "Vol%": round(s["AnnVol"]*100, 2) if s["AnnVol"] == s["AnnVol"] else np.nan,
            "Sharpe": round(s["Sharpe"], 3),
            "Sortino": round(s["Sortino"], 3),
            "MaxDD%": round(s["MaxDD"]*100, 2),
            "Calmar": round(s["Calmar"], 3),
            "AvgExp%": round(s["AvgExposure"]*100, 1),
            "TimeInMkt%": round(s["TimeInMkt"]*100, 1),
            "TurnoverAnn": round(s["TurnoverAnn"], 2),
            "CVaR5%": round(s["CVaR_5"]*100, 3) if s["CVaR_5"] == s["CVaR_5"] else np.nan,
        }

    comp = pd.DataFrame([
        row("SENTINEL_TECH10", s_sent),
        row("BUY_HOLD_TECH", s_bh),
        row("QQQ", s_qqq),
        row("^NDX", s_ndx),
        row("SPY", s_spy),
    ])

    print("\n" + "="*150)
    print("SENTINEL TECH 10.0 — COMPARATIVO PRINCIPAL (batir QQQ)")
    print("="*150)
    print(comp.to_string(index=False))

    # Robustez CV (no optimiza params; evalúa robustez del diseño)
    cv_rep = score_cv(prices, qqq_price, cfg, costs)
    print("\n" + "="*150)
    print("ROBUSTEZ (Purged Walk-Forward CV, test 2 años)")
    print("="*150)
    print(f"score={cv_rep['score']:.3f} | avgSharpe={cv_rep['avg_sharpe']:.3f} | stdSharpe={cv_rep['std_sharpe']:.3f} | worstDD={cv_rep['worst_dd']:.2%} | splits={cv_rep['splits']}")

    # Stress + Bootstrap
    stress = stress_report(prices, qqq_price, cfg, costs, STRESS_EPISODES)
    boot = moving_block_bootstrap(res["returns_net"], block=20, n_samples=500, seed=42)

    print("\n" + "="*150)
    print("STRESS EPISODES (burn_in=0)")
    print("="*150)
    print(stress.to_string(index=False))

    print("\nBOOTSTRAP blocks: DD p50={:.2f}%, DD p95(worst5%)={:.2f}%, P(DD<-50%)={:.2f}%".format(
        boot["dd_p50"]*100, boot["dd_p95"]*100, boot["ruin_prob_50dd"]
    ))

    # Exports
    os.makedirs("outputs", exist_ok=True)
    comp.to_csv("outputs/tech10_comparison.csv", index=False)
    stress.to_csv("outputs/tech10_stress.csv", index=False)

    # Plots
    if make_plots:
        plot_equity({
            "SENTINEL": res["equity"],
            "B&H_TECH": bh["equity"],
            "QQQ": qqq_eq,
            "^NDX": ndx_eq,
            "SPY": spy_eq
        }, "Equity Curves — Tech10")

        plot_drawdown(res["equity"], "Drawdown (Underwater) — Sentinel Tech10")
        plot_scales(res, "Exposure & Scales (Regime/Vol/Total) — Tech10")
        plot_turnover(res["turnover"], "Turnover (daily) — Tech10")
        plot_rolling_metrics(res["returns_net"], "Sentinel Tech10", window=126)
        plot_year_bars(res["returns_net"], "Sentinel Tech10")
        plot_year_bars(qqq_r, "QQQ")
        print_monthly_table(res["returns_net"], "Sentinel Tech10")
        print_monthly_table(qqq_r, "QQQ")
        plot_participation(res, qqq_r, "Sentinel Tech10", window=63)

    return {
        "comparison": comp,
        "cv_report": cv_rep,
        "stress": stress,
        "bootstrap": boot,
        "sentinel": res,
        "buy_hold": bh,
        "bench": {"QQQ": (qqq_r, qqq_eq), "NDX": (ndx_r, ndx_eq), "SPY": (spy_r, spy_eq)},
    }


if __name__ == "__main__":
    results = run_tech10(make_plots=True)

AttributeError: 'Tech10Config' object has no attribute 'burn_in'

: 